# Binary Signal generation

## All required imports

In [36]:
import pandas as pd
import os
import re
import numpy as np
from stockstats import wrap
from lightgbm import LGBMClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import torch
import torch.nn as nn

## Preprocessing

In [5]:
# Variables

col_names = ['timestamp', 'open', 'high', 'low', 'close', 'volume']


In [6]:
# Data extraction from Excel file
raw_df = pd.read_excel("../DATA_FOREX/1.EURUSD/Recent Data - (2023 - Latest)/RECENT_DATA_FILE_DUMP_EURUSD_M1.xlsx", header = None)

In [7]:
# Preparing clean dataframe from raw dataframe
df = raw_df[0].str.split(";", expand=True)
df.columns = col_names

# Removing extra data
clean_df  = df.drop(columns=['volume'])

# Convert price columns to float - Type of data checked
clean_df [['open', 'high', 'low', 'close']] = clean_df[['open', 'high', 'low', 'close']].astype(float)
clean_df ['timestamp'] = pd.to_datetime(clean_df ['timestamp'], format="%Y%m%d %H%M%S")

#Creating index from timestamp
clean_df.set_index('timestamp', inplace=True)


#Make sure data has no duplicate values
print(clean_df.index.is_unique)
duplicated_data = clean_df.index[clean_df.index.duplicated()]
clean_df = clean_df[~clean_df.index.duplicated(keep='first')]
print(clean_df.head())


False
                        open     high      low    close
timestamp                                              
2023-01-01 17:04:00  1.06970  1.06974  1.06970  1.06970
2023-01-01 17:05:00  1.06973  1.06978  1.06970  1.06971
2023-01-01 17:06:00  1.06966  1.06966  1.06966  1.06966
2023-01-01 17:08:00  1.06970  1.06974  1.06970  1.06974
2023-01-01 17:10:00  1.06975  1.06980  1.06972  1.06972


In [8]:
#Combine Text File generator
text_file_directory = "../DATA_FOREX/1.EURUSD/Recent Data - (2023 - Latest)/Text Files/"

def get_dir_files(text_file_directory: str) -> list: 
    """
    Syntax: os.listdir(path)   
    Parameters: path (optional) :  path of the directory  
    Return Type: This method returns the list of all files and directories in the specified path. The return type of this method is list. 
    """
    all_files = os.listdir(text_file_directory)
    return all_files

def combine_text_files(all_files: list):
    print(f"{all_files}")
    with open(f'{text_file_directory}/combined_txt_file.txt', 'a') as file:
        for a_file in all_files:
            print(f"{a_file}")
            with open(f'{text_file_directory}/{a_file}', 'r') as temp_file:
                file.write(temp_file.read() + '\n')
    return f'{text_file_directory}/combined_txt_file.txt'

all_files = get_dir_files(text_file_directory)
combine_text_file_path = combine_text_files(all_files)

print(f"New file created: {combine_text_file_path}")

['DAT_ASCII_EURUSD_M1_2023.txt', 'DAT_ASCII_EURUSD_M1_2024.txt', 'DAT_ASCII_EURUSD_M1_202501.txt', 'DAT_ASCII_EURUSD_M1_202502.txt', 'DAT_ASCII_EURUSD_M1_202503.txt', 'DAT_ASCII_EURUSD_M1_202504.txt']
DAT_ASCII_EURUSD_M1_2023.txt
DAT_ASCII_EURUSD_M1_2024.txt
DAT_ASCII_EURUSD_M1_202501.txt
DAT_ASCII_EURUSD_M1_202502.txt
DAT_ASCII_EURUSD_M1_202503.txt
DAT_ASCII_EURUSD_M1_202504.txt
New file created: ../DATA_FOREX/1.EURUSD/Recent Data - (2023 - Latest)/Text Files//combined_txt_file.txt


In [9]:
def parse_gap_report(file_path):
    gaps = []
    with open(file_path, 'r') as f:
        for line in f:
            match = re.match(r"Gap of (\d+)s found between (\d{14}) and (\d{14})\.", line)
            if match:
                duration = int(match.group(1))
                start = pd.to_datetime(match.group(2), format="%Y%m%d%H%M%S")
                end = pd.to_datetime(match.group(3), format="%Y%m%d%H%M%S")
                gaps.append({"start": start, "end": end, "duration_s": duration})
    return gaps

def handling_gaps(clean_df, combine_text_file_path):
    if combine_text_file_path:
        gaps = parse_gap_report(combine_text_file_path)
        # Forward fill small gaps (<= 300s) at 1-minute level
        clean_df = clean_df.asfreq('1T')
        # Flag large gaps (> 300s)
        for gap in gaps:
            #if gap['duration_s'] <= 300:
                #clean_df.loc[gap['start']:gap['end']] = clean_df.asfreq('1T', method='ffill')
            if gap['duration_s'] > 300:
                # Mark period as unreliable (e.g., set to NaN or flag)
                clean_df.loc[gap['start']:gap['end']] = None
                # missing_values = pd.DataFrame([gap['start'], gap['end']]) Have to improve in future
    else:
        # Forward fill all gaps if no gap report
        clean_df = clean_df.asfreq('1T', method='ffill')    
    return clean_df


#Execution of above funtions
clean_df = handling_gaps(clean_df, combine_text_file_path)
df_1min = clean_df.dropna()

### Labeling

In [10]:
# Labeling for recent 1m, 3m and 5m data at specific point in dataframe

df_1min = df_1min.copy()  

prev_close_1_min = df_1min['close'].shift(1)
conditions = [
    df_1min['close'] > prev_close_1_min,
    df_1min['close'] < prev_close_1_min
]
choices = [1, 2]

df_1min.loc[:, 'label_1min'] = np.select(conditions, choices, default=0)
df_1min.loc[df_1min.index[0], 'label_1min'] = 0


# 3 minutes candle
prev_close_3_min = df_1min['close'].shift(2)
conditions = [
    df_1min['close'] > prev_close_3_min,
    df_1min['close'] < prev_close_3_min
]
choices = [1, 2]

df_1min.loc[:, 'label_3min'] = np.select(conditions, choices, default=0)
df_1min.loc[df_1min.index[0], 'label_3min'] = 0



# 5 minutes candle
prev_close_3_min = df_1min['close'].shift(4)
conditions = [
    df_1min['close'] > prev_close_3_min,
    df_1min['close'] < prev_close_3_min 
]
choices = [1, 2]

df_1min.loc[:, 'label_5min'] = np.select(conditions, choices, default=0)
df_1min.loc[df_1min.index[0], 'label_5min'] = 0


# printing updated dataframe
df_1min.head(10)

,open,high,low,close,label_1min,label_3min,label_5min
timestamp,,,,,,,
2023-01-01 17:04:00,1.06970,1.06974,1.06970,1.06970,0,0,0
2023-01-01 17:05:00,1.06973,1.06978,1.06970,1.06971,1,0,0
2023-01-01 17:06:00,1.06966,1.06966,1.06966,1.06966,2,2,0
2023-01-01 17:08:00,1.06970,1.06974,1.06970,1.06974,1,1,0
2023-01-01 17:10:00,1.06975,1.06980,1.06972,1.06972,2,1,1
2023-01-01 17:11:00,1.06972,1.06972,1.06972,1.06972,0,2,1
2023-01-01 17:12:00,1.06975,1.06980,1.06975,1.06980,1,1,1
2023-01-01 17:13:00,1.07066,1.07066,1.06917,1.06943,2,2,2
2023-01-01 17:14:00,1.06937,1.06937,1.06899,1.06899,2,2,2


### Trend Calculation

Logic:
1.	Previous 3 Candle Trend: તમે છેલ્લાં 3 candles (1-minute, 3-minute, etc. timeframe મુજબ) લો.
2.	Candle Movement Count:
o	દરેક candle માટે જો close > open છે તો તેને UP (+1) ગણો.
o	જો close < open છે તો તેને DOWN (-1) ગણો.
o	Doji કે neutral candle હોય તો 0 ગણો (optional, depend on logic).
3.	Summing the Trend:
o	છેલ્લાં 3 candlesનું movement sum કરો.
o	Example: 
	Candle 1: UP (+1)
	Candle 2: DOWN (-1)
	Candle 3: UP (+1)
→ Sum = +1
→ Final Label = Neutral/Weak UP (as per your custom threshold)
4.	Label Assignment:
o	+2 or +3 → Label as UP (1)
o	-2 or -3 → Label as DOWN (0)
o	Anything between (-1 to +1) → Ignore or label as NEUTRAL (optional)

In [23]:
def calculate_trend(df, label_col, trend_col, window=3, exclude_current=False, neutral_value=int(0)):
    """
    Calculate trend from an existing label column using rolling sum of the last `window` directions.
    
    Parameters:
        df (pd.DataFrame): Input DataFrame.
        label_col (str): Name of existing label column (e.g., 'label_1min').
        trend_col (str): Name of new output trend column.
        window (int): Number of candles to use for trend (default=3).
        exclude_current (bool): If True, exclude current candle in trend calculation.
        neutral_value: Value for neutral trend (default=np.nan).

    Returns:
        pd.DataFrame: Original DataFrame with only one additional `trend_col`.
    """
    df = df.copy()

    # Map labels to direction values
    direction = pd.Series(
        np.select(
            [
                df[label_col] == 1,
                df[label_col] == 2
            ],
            [1, -1],
            default=0
        ),
        index=df.index
    )

    # Shift if current candle is to be excluded
    if exclude_current:
        direction = direction.shift(1)

    # Calculate rolling trend sum
    trend_sum = direction.rolling(window=window, min_periods=window).sum()

    # Assign trend label: 1 = UP, 0 = DOWN, else = neutral_value
    df[trend_col] = pd.Series(
        np.select(
            [
                trend_sum >= 2,
                trend_sum <= -2
            ],
            [1, -1],
            default=neutral_value
        ),
        index=df.index
    )

    return df[[trend_col]]

In [24]:
# First fix: assign only the Series, not a whole DataFrame
df_1min['trend_1min'] = calculate_trend(df_1min, label_col='label_1min', trend_col='trend_1min')['trend_1min']
df_1min['trend_3min'] = calculate_trend(df_1min, label_col='label_3min', trend_col='trend_3min')['trend_3min']
df_1min['trend_5min'] = calculate_trend(df_1min, label_col='label_5min', trend_col='trend_5min')['trend_5min']

df_1min.head(20)

,open,high,low,close,label_1min,label_3min,label_5min,trend_1min,trend_3min,trend_5min
timestamp,,,,,,,,,,
2023-01-01 17:04:00,1.06970,1.06974,1.06970,1.06970,0,0,0,0,0,0
2023-01-01 17:05:00,1.06973,1.06978,1.06970,1.06971,1,0,0,0,0,0
2023-01-01 17:06:00,1.06966,1.06966,1.06966,1.06966,2,2,0,0,0,0
2023-01-01 17:08:00,1.06970,1.06974,1.06970,1.06974,1,1,0,0,0,0
2023-01-01 17:10:00,1.06975,1.06980,1.06972,1.06972,2,1,1,0,0,0
2023-01-01 17:11:00,1.06972,1.06972,1.06972,1.06972,0,2,1,0,0,1
2023-01-01 17:12:00,1.06975,1.06980,1.06975,1.06980,1,1,1,0,0,1
2023-01-01 17:13:00,1.07066,1.07066,1.06917,1.06943,2,2,2,0,0,0
2023-01-01 17:14:00,1.06937,1.06937,1.06899,1.06899,2,2,2,0,0,0


### Indicators

In [32]:
df_1min = df_1min.reset_index()  # stockstats requires 'timestamp' as a column

# Wrap with stockstats
sdf = wrap(df_1min)

# Compute indicators
sdf['close_5_ema']     # EMA(5)
sdf['close_10_ema']    # EMA(10)
sdf['rsi_14']          # RSI(14)
sdf['macdh']           # MACD Histogram
sdf['adx']             # ADX
sdf['atr']             # ATR
sdf['boll_ub']         # Bollinger Upper Band
sdf['boll_lb']         # Bollinger Lower Band
sdf['boll_width'] = sdf['boll_ub'] - sdf['boll_lb']  # Bollinger Band Width

# Manually compute Candle Body/Wick Ratio
sdf['body'] = abs(sdf['close'] - sdf['open'])
sdf['wick'] = sdf['high'] - sdf['low']
sdf['body_wick_ratio'] = sdf['body'] / sdf['wick'].replace(0, 1e-9)

# Optional: drop intermediate body/wick columns if you don't want them
# sdf.drop(columns=['body', 'wick'], inplace=True)


final_df = sdf[[
    'timestamp', 'open', 'high', 'low', 'close',
    'close_5_ema', 'close_10_ema', 'rsi_14', 'macdh',
    'adx', 'atr', 'boll_width', 'body_wick_ratio', 'label_1min', 'label_3min', 'label_5min',
    'trend_1min', 'trend_3min', 'trend_5min'
]]

print(final_df.head(10))

            timestamp     open     high      low    close  close_5_ema  \
0 2023-01-01 17:04:00  1.06970  1.06974  1.06970  1.06970     1.069700   
1 2023-01-01 17:05:00  1.06973  1.06978  1.06970  1.06971     1.069706   
2 2023-01-01 17:06:00  1.06966  1.06966  1.06966  1.06966     1.069684   
3 2023-01-01 17:08:00  1.06970  1.06974  1.06970  1.06974     1.069707   
4 2023-01-01 17:10:00  1.06975  1.06980  1.06972  1.06972     1.069712   
5 2023-01-01 17:11:00  1.06972  1.06972  1.06972  1.06972     1.069715   
6 2023-01-01 17:12:00  1.06975  1.06980  1.06975  1.06980     1.069745   
7 2023-01-01 17:13:00  1.07066  1.07066  1.06917  1.06943     1.069636   
8 2023-01-01 17:14:00  1.06937  1.06937  1.06899  1.06899     1.069415   
9 2023-01-01 17:15:00  1.06788  1.06788  1.06788  1.06788     1.068894   

   close_10_ema      rsi_14         macdh         adx       atr  boll_width  \
0      1.069700         NaN  0.000000e+00         NaN  0.000040         NaN   
1      1.069705  100.000000

### Defining training variables and target variables 

# Reruired output format gien by client
{
  "pair": "EUR/USD",
  "timeframe": "3m",
  "signal": "UP",
  "confidence": 0.734
}

Queries: 
What are the target variables should we aim for?
Should we only use trend as a target for all 1 min, 2 min and 3 min?

### Feature Elimination

In [33]:
df_feat = final_df

target_col = "trend_1min"
X = df_feat.drop(columns=[target_col])
y = df_feat[target_col]

In [34]:
def remove_highly_correlated_features(X, threshold=0.8):
    """
    Removes features that are highly correlated with each other (above the given threshold).
    
    Parameters:
        df (pd.DataFrame): Input DataFrame with numerical features only.
        threshold (float): Correlation threshold above which features are considered redundant.

    Returns:
        pd.DataFrame: DataFrame with reduced features.
        list: List of dropped features.
    """
    df = X
    corr_matrix = df.corr().abs()  # absolute correlation
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
    
    reduced_df = df.drop(columns=to_drop)
    
    return reduced_df, to_drop

X_reduced, dropped_features = remove_highly_correlated_features(X, threshold=0.8)

print("Dropped features due to high correlation:")
print(dropped_features)

Dropped features due to high correlation:
['high', 'low', 'close', 'close_5_ema', 'close_10_ema']


In [35]:
X_reduced.head(20)

,timestamp,open,rsi_14,macdh,adx,atr,boll_width,body_wick_ratio,label_1min,label_3min,label_5min,trend_3min,trend_5min
0,2023-01-01 17:04:00,1.06970,NaN,0.000000e+00,NaN,0.000040,NaN,0.000000,0,0,0,0,0
1,2023-01-01 17:05:00,1.06973,100.000000,9.971510e-08,100.000000,0.000061,0.000028,0.250000,1,0,0,0,0
2,2023-01-01 17:06:00,1.06966,15.662651,-8.198929e-07,43.827160,0.000057,0.000106,0.000000,2,2,0,0,0
3,2023-01-01 17:08:00,1.06970,65.621458,9.404598e-07,47.055883,0.000063,0.000132,1.000000,1,1,0,0,0
4,2023-01-01 17:10:00,1.06975,56.595403,9.430543e-07,54.237264,0.000067,0.000119,0.375000,2,1,1,0,0
5,2023-01-01 17:11:00,1.06972,56.595403,8.446537e-07,58.242083,0.000054,0.000109,0.000000,0,2,1,0,1
6,2023-01-01 17:12:00,1.06975,73.502912,3.420649e-06,64.258935,0.000058,0.000170,1.000000,1,1,1,0,1
7,2023-01-01 17:13:00,1.07066,24.999525,-8.932152e-06,73.948912,0.000287,0.000441,0.825503,2,2,2,0,0
8,2023-01-01 17:14:00,1.06937,13.549214,-3.162344e-05,71.401112,0.000309,0.001014,1.000000,2,2,2,0,0
9,2023-01-01 17:15:00,1.06788,6.037049,-8.614706e-05,55.106916,0.000419,0.002386,0.000000,2,2,2,-1,-1


### Windows creation

In [39]:
def create_sequences(data, target_col, window_size=10):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data.iloc[i:i+window_size][data.columns].values)
        y.append(data.iloc[i+window_size][target_col])
    return np.array(X), np.array(y)


### Train Test Split

In [41]:

X_seq, y_seq = create_sequences(X, target_col=y, window_size=10)
X_train, X_test, y_train, y_test = train_test_split(X_seq, y_seq, test_size=0.2, shuffle=False)


MemoryError: Unable to allocate 6.14 MiB for an array with shape (804717,) and data type int64

### LSTM Model

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers=1, output_size=1):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.linear = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]  # Take last time step
        return self.linear(out)


### Training Model

In [ ]:
model = LSTMModel(input_size=X_train.shape[2], hidden_size=64)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Convert to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

# Training loop
for epoch in range(50):
    model.train()
    output = model(X_train_tensor)
    loss = criterion(output.squeeze(), y_train_tensor)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if epoch % 10 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")


In [ ]:
model.eval()
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
with torch.no_grad():
    predictions = model(X_test_tensor).squeeze()

# Optional: Evaluate accuracy or RMSE
